In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(api_key=groq_api_key, model="openai/gpt-oss-120b")

In [44]:
## Loader, Vector Store, Retriever

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

from langchain_chroma import Chroma
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1373.64it/s]


In [45]:
import bs4

loader = WebBaseLoader(
    web_path = "https://lilianweng.github.io/posts/2023-01-10-inference-optimization/",
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            class_ = ["post-title", "post-content"]
        )
    )
)

docs = loader.load()

In [46]:
docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-01-10-inference-optimization/'}, page_content='\n      Large Transformer Model Inference Optimization\n    [Updated on 2023-01-24: add a small section on Distillation.]\nLarge transformer models are mainstream nowadays, creating SoTA results for a variety of tasks. They are powerful but very expensive to train and use. The extremely high inference cost, in both time and memory, is a big bottleneck for adopting a powerful transformer for solving real-world tasks at scale.\nWhy is it hard to run inference for large transformer models? Besides the increasing size of SoTA models, there are two main factors contributing to the inference challenge (Pope et al. 2022):\n\nLarge memory footprint. Both model parameters and intermediate states are needed in memory at inference time. For example,\n\nThe KV cache should be stored in memory during decoding time; E.g. For a batch size of 512 and context length of 2048, the KV cach

In [47]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

texts = text_splitter.split_documents(docs)

In [48]:
vectorstore = Chroma.from_documents(documents=texts, embedding=embeddings)

In [49]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000017D82D2BD90>, search_kwargs={'k': 3})

In [50]:
system_prompt = (
"You are a helpful assistant for answering questions about the following context: "
"\n\n"
"{context}""Multiturn Conversational Chatbot Projects"
"\n\n"
"if you are unsure of the answer, say you don't know. " 
"Answer in three to four lines in a friendly way."
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [51]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

contextualize_q_system_prompt = (
    "Given a chat history and the latest user questions "
    "Which might reference context in the chat history "
    "forumulate a standalone question that can be understood "
    "without the need of the chat history. DO NOT answer the question, "
    "just reformulate it if needed otherwise return the question as is. "
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}")
    ]
)

In [52]:
history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_q_prompt
)

In [53]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}")
    ]
)

In [54]:
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [55]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain, 
    get_session, 
    input_messages_key="input", 
    history_messages_key="chat_history", 
    output_messages_key="answer"    
)

e:\TCS\Langchain\venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [56]:
config = {
    "configurable": {
        "session_id": "chat1"
    }
}

response = conversational_rag_chain.invoke({"input": "What is Large Model Inference?"}, config=config)
response["answer"]

'Large Model Inference is the step where a trained, usually very big (e.g., transformer‑based) model is used to generate predictions or outputs on new data. Because the model’s parameters and intermediate activations are huge, it demands a lot of GPU memory, FLOPs, and time. Optimizing this stage means cutting memory footprints, reducing compute, and lowering latency so the model can serve real‑world tasks efficiently.'

In [57]:
config = {
    "configurable": {
        "session_id": "chat1"
    }
}

response = conversational_rag_chain.invoke({"input": "What are the challenges?"}, config=config)
response["answer"]

'The main challenges are:\n\n1️⃣ **Huge memory needs** – the model’s weights and its intermediate activations must fit in GPU RAM, which quickly exceeds a single device.  \n2️⃣ **Massive compute cost** – billions of FLOPs per token make inference slow and expensive.  \n3️⃣ **Latency & scalability** – high per‑token latency hurts real‑time apps and limits serving many users simultaneously.  \n\nTackling these issues requires clever memory‑saving tricks, model‑size reductions, and speed‑up techniques.'